# 🚀 Ausblick: Die Auswirkung der LLM-Nachkorrektur auf die Suche nach Textspuren der Spanischen Grippe <!-- Outlook: LLM postcorrection effect on the search for Spanish Flu -->


Nachdem wir unsere Analyse abgeschlossen haben, kommen wir nun zurück zur Frage der LLM-Nachkorrektur. Was würde passieren, wenn wir unser Korpus mit LLMs nachkorrigieren würden? Wir haben es an einer Datei (SNP2719372X-19181012-0-0-0-0.pdf) getestet und die gesamte OCR-Datei über die GPT-4-API laufen lassen. Hier sind die beiden Segmente derselben Seite dargestellt, die den Unterschied vor (links) und nach (rechts) der LLM-Nachkorrektur veranschaulichen:

<!-- Now that we have done our analysis, let us get back to the question of LLM postcorrection. What would happen, if we post-corrected our corpus with LLMs? We tested it on one file, running the whole OCR-ed file through GPT-4 API. Here are the two segments of the same page that demonstrate the difference before (left) and after (right) the LLM postcorrection:-->

![llm_vs_original](../assets/images/llm_vs_original.jpg)

<!-- Let us now see if this affected the count of Grippe-related words in that file: -->

Sehen wir uns nun an, ob sich dies auf die Anzahl der Grippe-bezogenen Wörter in dieser Datei auswirkt:

## Definieren der Wortliste (erneut):

In [ ]:
grippe_list = ['Influenza',
'Grippe',
'Grippeepidemie',
'Grippewelle',
'Grippekranke',              
'Grippepandemie',
'Lungenentzündung',
'Krankheitswelle',
'Seuchenzug',
'Krankheitsausbruch',
'Fieberanfall',
'Schüttelfrost',
'Atemnot',
'Körpererschöpfung',
'Genesungszeit',
'Ansteckungsgefahr',
'Seuchenschutz',
'Desinfektionsmittel',
'Schutzmaske',
'Krankenstation',
'Isolationsstation',
'Sanitätsdienst',
'Krankheitsverlauf',
'Todesopfer',
'Krankheitssymptom',
'Erkrankungsfall',
'Lungeninfektion']

## Lesen der Anmerkungen der beiden Dateien (Original und nachträglich korrigiert) <!-- Reading the annotations of the two files (original and post-corrected) -->

In [ ]:
!pip install requests

import pandas as pd
import requests
import sys

Bevor wir Daten herunterladen, definieren wir eine kleine Hilfsfunktion `download_file`. Sie lädt eine Datei plattformunabhängig – also auch unter Windows – aus dem Internet in einen Zielordner herunter und ersetzt damit das Kommando `wget`, das nicht auf allen Systemen (z.B. Windows) nativ verfügbar ist.

In [ ]:
# helper: download a single file (cross-platform replacement for `! wget -P`)
from pathlib import Path

def download_file(url, target_dir):
    """Download the file at `url` into `target_dir`, keeping its original name."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / url.split("/")[-1]
    response = requests.get(url)
    response.raise_for_status()
    target_path.write_bytes(response.content)
    return target_path

In [ ]:
if not Path("../data/csv/SNP2719372X-19181012-0-0-0-0.csv").exists():
    download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/main/data/csv/SNP2719372X-19181012-0-0-0-0.csv", "../data/csv")
if not Path("../data/aux_postcorr/SNP2719372X-19181012-0-0-0-0_llm_postcorr.csv").exists():
    download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/main/data/aux_postcorr/SNP2719372X-19181012-0-0-0-0_llm_postcorr.csv", "../data/aux_postcorr")

In [ ]:
original = pd.read_csv('../data/csv/SNP2719372X-19181012-0-0-0-0.csv')
post_llm = pd.read_csv('../data/aux_postcorr/SNP2719372X-19181012-0-0-0-0_llm_postcorr.csv')

## Zählen der Wörter der Liste für jede Datei <!-- Counting the words of the list for each file -->

In [ ]:
def count_words(annotation, wordlist):
    result = annotation.query(f'Lemma.isin({wordlist})')
    return result.shape[0]

In [ ]:
count_words(original, grippe_list)

In [ ]:
count_words(post_llm, grippe_list)

LLM verbessert in diesem Fall also nicht nur das Erscheinungsbild der Texte, sondern hilft auch dabei, einige Erwähnungen von Grippe-bezogenen Wörtern wiederherzustellen.
<!-- So LLM in this case improves not only the appearance of the texts, but also actually helps to recover some mentions of Grippe-related words -->